# 02 · M2: ai_parse_document + ai_extract

Esquema = nombres de campo de DocILE (21 de información clave + 12 de ítem de línea), sin ejemplos (zero-shot).
Celdas en orden. **Celda 2 procesa 1 documento** para revisar la salida antes de correr los 20.


In [ ]:
%sql
-- Celda 1: parsear los 20 PDF del piloto (una vez)
CREATE OR REPLACE TABLE workspace.invoice_extraction_benchmark.m2_parsed AS
SELECT
  regexp_extract(path, '([^/]+)\.pdf$', 1) AS docid,
  path,
  ai_parse_document(content, map('version', '2.0')) AS parsed,
  current_timestamp() AS parsed_at
FROM read_files('/Volumes/workspace/invoice_extraction_benchmark/docile_pdfs/', format => 'binaryFile');

SELECT count(*) AS documentos FROM workspace.invoice_extraction_benchmark.m2_parsed;

In [ ]:
%sql
-- Celda 2: extraer 1 documento (prueba)
SELECT
  docid,
  ai_extract(
    parsed,
    '{"document_id": {"type": "string", "description": "Invoice/document number"}, "date_issue": {"type": "string", "description": "Issue date"}, "date_due": {"type": "string", "description": "Due date"}, "order_id": {"type": "string", "description": "Order number referenced by the document"}, "customer_id": {"type": "string", "description": "Customer/account number"}, "customer_order_id": {"type": "string", "description": "Customer purchase order number"}, "vendor_name": {"type": "string", "description": "Issuer/vendor name"}, "vendor_address": {"type": "string", "description": "Issuer/vendor address"}, "vendor_tax_id": {"type": "string", "description": "Vendor tax ID"}, "vendor_email": {"type": "string", "description": "Vendor email"}, "customer_billing_name": {"type": "string", "description": "Bill-to name"}, "customer_billing_address": {"type": "string", "description": "Bill-to address"}, "customer_delivery_name": {"type": "string", "description": "Ship-to name"}, "customer_delivery_address": {"type": "string", "description": "Ship-to address"}, "amount_total_net": {"type": "string", "description": "Total before tax"}, "amount_total_tax": {"type": "string", "description": "Total tax"}, "amount_total_gross": {"type": "string", "description": "Total including tax"}, "amount_due": {"type": "string", "description": "Amount due / balance"}, "amount_paid": {"type": "string", "description": "Amount already paid"}, "currency_code_amount_due": {"type": "string", "description": "Currency of the amount due"}, "payment_terms": {"type": "string", "description": "Payment terms"}, "line_items": {"type": "array", "description": "One object per table row", "items": {"type": "object", "properties": {"line_item_position": {"type": "string", "description": "Row number"}, "line_item_date": {"type": "string", "description": "Date of the item/service"}, "line_item_code": {"type": "string", "description": "Item code/SKU"}, "line_item_description": {"type": "string", "description": "Item description"}, "line_item_quantity": {"type": "string", "description": "Quantity"}, "line_item_units_of_measure": {"type": "string", "description": "Unit of measure"}, "line_item_unit_price_net": {"type": "string", "description": "Unit price before tax"}, "line_item_unit_price_gross": {"type": "string", "description": "Unit price"}, "line_item_amount_net": {"type": "string", "description": "Line amount before tax"}, "line_item_amount_gross": {"type": "string", "description": "Line amount"}, "line_item_currency": {"type": "string", "description": "Currency"}, "line_item_person_name": {"type": "string", "description": "Person name in the row"}}}}}',
    map('version', '2.1',
        'enableCitations', 'true',
        'enableConfidenceScores', 'true',
        'instructions', 'Extract each value exactly as printed in the document, without reformatting dates or numbers. Omit fields that are not present.')
  ) AS extracted
FROM workspace.invoice_extraction_benchmark.m2_parsed
WHERE docid = 'f7b33ee579c24727a2a8923f';